# ViSpatialQA — E7: thêm nguồn L2 (Qwen3-30B-A3B-Thinking-2507) cho Human

**Vì sao:** sau E4/E6, phần còn thiếu nằm ở Human và là lỗi *đọc/suy luận*: trên 613 câu có 20 câu FR, 20 câu CO, 7 câu YN mà **không nguồn nào đúng**, nên chỉnh cách gộp không cứu được. Giải mã FR theo từng trục (CPU) đã thử trên 613 câu: không hơn bỏ phiếu hiện tại, nên không dùng.

**Làm gì:** chạy nhánh L-CoT với một model thinking khác họ, mạnh hơn Qwen3-32B trên suy luận (bản 2507, ~61GB bf16, vừa A100 80GB), trên Human train + dev + test (745 câu, n=8 mẫu mỗi câu). Kết quả là nguồn **LCOT2**. Sau đó so ba biến thể trên 613 câu ngoài mẫu và ghi file nộp của biến thể tốt nhất:
- `base`: F + S + L + P (như E6, không có LCOT2)
- `+L2`: thêm LCOT2 như một nhánh riêng (M)
- `L2 thay L`: LCOT2 thay LCOT

**Cần có trên Drive:** `outputs/predictions_e6/` (từ E6) và `outputs/e4/trdev_LPX/F+S+L+P/auto_submission.json`.
**Chỉ cần GPU cho ô 4 (smoke test và chạy L2).** Mọi thứ sau đó chạy CPU. Thời gian dự kiến cho L2: khoảng 40–110 phút trên A100 (Qwen3-32B mất ~90 phút cho cùng khối lượng; bản A3B MoE nhanh hơn, nhưng đây là ước lượng, chưa đo).

## 1. CONFIG

In [1]:
CONFIG = dict(
    REPO_URL="https://github.com/lightanddark0/VLSP-Task5.git",
    BRANCH="feat/e6-fr-dk-groupb",
    WORK_DIR="/content/VLSP-Task5",
    DRIVE_DATA_DIR="/content/drive/MyDrive/VLSP2026/Data",
    DRIVE_BACKUP_DIR="/content/drive/MyDrive/VLSP2026/outputs",
    BEST_E4_DIR="e4/trdev_LPX/F+S+L+P",                # lấy auto_submission.json từ đây
    P_SOURCE="LPY",                                    # nguồn LP của E6 (biến thể tốt nhất trên 613 câu)
    L2=dict(model="Qwen/Qwen3-30B-A3B-Thinking-2507", source="LCOT2", n=4, max_tokens=8192, max_model_len=12288,
            max_num_seqs=96, gpu_memory_utilization=0.92, fewshot_k=2),
    SMOKE_LIMIT=12,                                    # số câu Human dev chạy thử trước khi chạy thật (0 = bỏ qua)
    HUMAN_YN_DK_FINAL="keep",                          # GIỮ DK: Human test có câu DK (E4 giữ DK được 47/51 YN, bản "no" chỉ 44/51)
    SMOKE_N=4,                                         # số mẫu mỗi câu ở smoke test
    C_GRID="0.1,1,10",
)

## 2. Chuẩn bị

In [2]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
from IPython.display import Markdown, display

C = CONFIG
WORK, DRIVE_OUT = Path(C["WORK_DIR"]), Path(C["DRIVE_BACKUP_DIR"])
START = time.time()
os.environ["VLLM_USE_DEEP_GEMM"] = "0"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
os.environ.setdefault("WANDB_MODE", "disabled")


def sh(cmd, capture=False):
    """Chạy lệnh trong WORK_DIR và in output ngay (qua pipe, nên Colab hiện được cả lỗi của tiến trình con)."""
    import codecs
    cmd = [str(part) for part in cmd]
    print("$", " ".join(cmd), flush=True)
    started = time.time()
    process = subprocess.Popen(cmd, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               env=dict(os.environ, PYTHONUNBUFFERED="1"))
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    tail = ""
    while chunk := os.read(process.stdout.fileno(), 4096):
        text = decoder.decode(chunk)
        tail = (tail + text)[-6000:]
        if not capture:
            sys.stdout.write(text); sys.stdout.flush()
    code = process.wait()
    if capture and code:
        print(tail)
    print(f"[exit {code}, {(time.time() - started) / 60:.1f} phút]", flush=True)
    if code:
        raise RuntimeError(f"Lệnh lỗi (exit {code}); xem output phía trên")
    return tail


if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
if (WORK / ".git").exists():
    for args in (["fetch", "--quiet", "origin", C["BRANCH"]], ["checkout", "--quiet", "-B", C["BRANCH"], f"origin/{C['BRANCH']}"]):
        subprocess.run(["git", "-C", str(WORK), *args], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", "--branch", C["BRANCH"], C["REPO_URL"], str(WORK)], check=True)
os.chdir(WORK)
print("Code:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"], check=True)
for name in ["human_train.json", "auto_train.json", "human_public_test.json", "auto_public_test.json"]:
    if not (WORK / "Data" / name).exists():
        shutil.copy2(Path(C["DRIVE_DATA_DIR"]) / name, WORK / "Data" / name)
if not (WORK / "Data/splits/human_dev.json").exists():
    sh(["python", "make_splits.py", "--from-manifest"])

PRED, OUT = WORK / "outputs/predictions_e7", WORK / "outputs/e7"
OUT.mkdir(parents=True, exist_ok=True)
for folder in ("predictions_e7", "predictions_e6"):             # resume E7 trước, rồi lấy nền từ E6
    if (DRIVE_OUT / folder).exists():
        for path in (DRIVE_OUT / folder).rglob("*.jsonl"):
            target = PRED / path.relative_to(DRIVE_OUT / folder)
            if not target.exists() and ".raw." not in path.name and "folds" not in str(path.relative_to(DRIVE_OUT / folder)):
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, target)
auto_sub = DRIVE_OUT / C["BEST_E4_DIR"] / "auto_submission.json"
assert auto_sub.exists(), f"Thiếu {auto_sub}"
P = C["P_SOURCE"]
need = ["F2C/human_train.jsonl", "F2C/human_dev.jsonl", "F2C/human_test.jsonl", "F1/human_test.jsonl", "S/human_test.jsonl",
        "LCOT/human_train.jsonl", "LCOT/human_test.jsonl", f"{P}/human_train.jsonl", f"{P}/human_test.jsonl"]
assert all((PRED / n).exists() for n in need), f"Thiếu {[n for n in need if not (PRED / n).exists()]}: cần predictions_e6 trên Drive."


def to_drive():
    for folder in ("predictions_e7", "e7"):
        if (WORK / "outputs" / folder).exists():
            shutil.copytree(WORK / "outputs" / folder, DRIVE_OUT / folder, dirs_exist_ok=True)


SPLIT_FILES = {"train": "Data/splits/human_train.json", "dev": "Data/splits/human_dev.json", "test": "Data/human_public_test.json"}
print(f"Chuẩn bị xong sau {(time.time() - START) / 60:.1f} phút. Nguồn LP: {P}")

Mounted at /content/drive
Code: 1dbb15c E6 notebook: always use fr_unknown=abstain for LPY; grid only lenient
$ python make_splits.py --from-manifest
Rebuilt human split from manifest (7 dev stories)
Rebuilt auto split from manifest (1308 dev stories)
[exit 0, 0.1 phút]
Chuẩn bị xong sau 0.9 phút. Nguồn LP: LPY


## 3. Cài vLLM (chỉ lần đầu mỗi phiên)

In [3]:
if subprocess.run([sys.executable, "-c", "import vllm"], capture_output=True).returncode != 0:
    sh(["pip", "install", "-q", "vllm"])
    subprocess.run(["pip", "uninstall", "-y", "-q", "torchaudio", "torchao"])
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)

$ pip install -q vllm
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 117.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 117.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

CompletedProcess(args='nvidia-smi --query-gpu=name,memory.total --format=csv', returncode=0)

## 4. Smoke test rồi chạy L2 trên Human train + dev + test
Smoke test chạy `SMOKE_LIMIT` câu Human dev vào thư mục riêng và in tỷ lệ mẫu hỏng và độ chính xác so với LCOT trên cùng câu. Nếu tỷ lệ abstain cao (nhiều mẫu bị cắt hoặc không có dòng `ĐÁP ÁN:`), dừng lại và báo lại trước khi chạy tiếp.

In [4]:
from spartqa.data import read_json
from spartqa.metrics import answer_matches
from spartqa.predictions import iter_questions, read_predictions

L2 = dict(C["L2"])
src = L2["source"]
# GPU nhỏ (< 60GB, ví dụ A100 40GB): bản bf16 (~61GB) không vừa -> dùng bản FP8 (~31GB) với ít chuỗi song song hơn.
gpu_gb = int(subprocess.run("nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits", shell=True,
                            capture_output=True, text=True).stdout.split()[0]) / 1024
if gpu_gb < 60 and not L2["model"].endswith("-FP8"):
    L2.update(model=L2["model"] + "-FP8", max_num_seqs=8, max_model_len=10240, max_tokens=7168, gpu_memory_utilization=0.94)
print(f"GPU {gpu_gb:.0f} GB -> model {L2['model']}, max_num_seqs {L2['max_num_seqs']}, max_model_len {L2['max_model_len']}")

RUN_L2 = r'''"""Chạy llm_cot.py với bộ parse khoan dung: bản 2507-Thinking không in thẻ <think> mở và có thể viết đáp án dạng khác."""
import re, sys
import spartqa.prompting as P

_orig, _LIST = P.parse_cot_answer, re.compile(r"\[[^\[\]]*\]")
_MARK = re.compile(r"đáp\s*án(?:\s*cuối\s*cùng)?\s*[:：]?[\s*`]*", re.IGNORECASE)


def tolerant(text, payload):
    got = _orig(text, payload)
    if got is not None:
        return got
    visible = text.rsplit("</think>", 1)[1] if "</think>" in text else text[-1500:]
    marks = list(_MARK.finditer(visible))
    candidates = _LIST.findall(visible[marks[-1].end():])[:1] if marks else _LIST.findall(visible)[-1:]
    if marks and not candidates:                       # đáp án không có ngoặc: ĐÁP ÁN: Yes / ĐÁP ÁN: 0, 5
        tail = visible[marks[-1].end():][:40]
        word = re.match(r"\W*(yes|no|dk|có|không)\b", tail, re.IGNORECASE)
        digits = re.match(r"\W*(\d(?:\s*,\s*\d)*)", tail)
        candidates = ['["%s"]' % word.group(1)] if word else (["[%s]" % digits.group(1)] if digits else [])
    for candidate in candidates:
        answer = P.parse_answer(candidate, payload)
        if answer is not None:
            return answer
    return None


P.parse_cot_answer = tolerant
import llm_cot
raise SystemExit(llm_cot.main(sys.argv[1:]))
'''
(WORK / "run_l2.py").write_text(RUN_L2, encoding="utf-8")


def l2_cmd(jobs, extra=(), n=None):
    return ["python", "run_l2.py", "--save-raw", "--source", src, "--model-name", L2["model"], "--fallback-model", "", "--n", n or L2["n"],
            "--max-tokens", L2["max_tokens"], "--max-model-len", L2["max_model_len"], "--max-num-seqs", L2["max_num_seqs"],
            "--fewshot-k", L2["fewshot_k"], "--gpu-memory-utilization", L2["gpu_memory_utilization"], "--seed", 42,
            *extra, *sum((["--job", f"human,{SPLIT_FILES[s]},{path}"] for s, path in jobs), [])]


if C["SMOKE_LIMIT"]:
    smoke = OUT / "smoke/human_dev.jsonl"
    smoke.parent.mkdir(parents=True, exist_ok=True)
    smoke.unlink(missing_ok=True)
    sh(l2_cmd([("dev", smoke)], ["--limit", C["SMOKE_LIMIT"], "--run-name", "E7-L2-smoke"], n=C["SMOKE_N"]))
    mine, base = read_predictions(smoke), read_predictions(PRED / "LCOT/human_dev.jsonl")
    gold = {key: q for key, _, q in iter_questions(read_json(SPLIT_FILES["dev"]))}
    n = len(mine)
    abstain = sum(r["answer"] is None for r in mine.values())
    ok = sum(answer_matches(r["q_type"], r["answer"], gold[k]["answer"]) for k, r in mine.items() if r["answer"] is not None)
    ok_base = sum(answer_matches(base[k]["q_type"], base[k]["answer"], gold[k]["answer"]) for k in mine if k in base and base[k]["answer"] is not None)
    print(f"Smoke: {n} câu, abstain {abstain}, đúng {ok}; LCOT trên cùng câu đúng {ok_base}")
    if not n or abstain / n > 0.15:
        for line in smoke.with_suffix(".raw.jsonl").read_text(encoding="utf-8").splitlines()[:3]:
            record = json.loads(line)
            for text in sum(record["texts"].values(), [])[:2]:
                print("=" * 100)
                print(f"key={record['key']} dài={len(text)} | có </think>: {'</think>' in text} | có <think>: {'<think>' in text} | có 'ĐÁP ÁN': {'ĐÁP ÁN' in text}")
                print("... " + text[-700:])
    assert n and abstain / n <= 0.15, "Quá nhiều câu abstain: kiểm tra mẫu hỏng (finish_reason=length hoặc không có ĐÁP ÁN)."

GPU 80 GB -> model Qwen/Qwen3-30B-A3B-Thinking-2507, max_num_seqs 96, max_model_len 12288
$ python run_l2.py --save-raw --source LCOT2 --model-name Qwen/Qwen3-30B-A3B-Thinking-2507 --fallback-model  --n 4 --max-tokens 8192 --max-model-len 12288 --max-num-seqs 96 --fewshot-k 2 --gpu-memory-utilization 0.92 --seed 42 --limit 12 --run-name E7-L2-smoke --job human,Data/splits/human_dev.json,/content/VLSP-Task5/outputs/e7/smoke/human_dev.jsonl
W&B tracking disabled (no wandb, no WANDB_API_KEY, or WANDB_MODE=disabled).
INFO 10-02 03:43:46 [api_utils.py:286] non-default args: {'seed': 42, 'max_model_len': 12288, 'enable_prefix_caching': True, 'max_num_seqs': 96, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-30B-A3B-Thinking-2507'}
INFO 10-02 03:44:05 [model.py:692] Resolved architecture: Qwen3MoeForCausalLM
INFO 10-02 03:44:05 [model.py:2030] Using max model len 12288
INFO 10-02 03:44:05 [scheduler.py:288] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 10-02 03:44:05 [ker

In [5]:
t0 = time.time()
for splits in (("dev", "test"), ("train",)):          # sao lưu lên Drive sau mỗi lượt, để Colab ngắt thì chạy lại là tiếp tục
    sh(l2_cmd([(s, PRED / src / f"human_{s}.jsonl") for s in splits], ["--run-name", f"E7-L2-{'-'.join(splits)}"]))
    to_drive()
print(f"L2 xong sau {(time.time() - t0) / 60:.1f} phút")

$ python run_l2.py --save-raw --source LCOT2 --model-name Qwen/Qwen3-30B-A3B-Thinking-2507 --fallback-model  --n 4 --max-tokens 8192 --max-model-len 12288 --max-num-seqs 96 --fewshot-k 2 --gpu-memory-utilization 0.92 --seed 42 --run-name E7-L2-dev-test --job human,Data/splits/human_dev.json,/content/VLSP-Task5/outputs/predictions_e7/LCOT2/human_dev.jsonl --job human,Data/human_public_test.json,/content/VLSP-Task5/outputs/predictions_e7/LCOT2/human_test.jsonl
W&B tracking disabled (no wandb, no WANDB_API_KEY, or WANDB_MODE=disabled).
INFO 10-02 03:52:00 [api_utils.py:286] non-default args: {'seed': 42, 'max_model_len': 12288, 'enable_prefix_caching': True, 'max_num_seqs': 96, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-30B-A3B-Thinking-2507'}
INFO 10-02 03:52:00 [model.py:692] Resolved architecture: Qwen3MoeForCausalLM
INFO 10-02 03:52:00 [model.py:2030] Using max model len 12288
INFO 10-02 03:52:00 [scheduler.py:288] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO

## 5. So sánh ba biến thể trên 613 câu Human (ngoài mẫu, cùng fold story)

In [6]:
sh(["python", "crossfit.py", "combine", "--pred-dir", PRED, "--name", "trdev"])
auto_cv = 0.9873
BASE = f"F=F2C,F1 S=S L=LCOT P={P}"
VARIANTS = {"base (không L2)": BASE, "+L2 (nhánh M)": BASE + f" M={src}", "L2 thay L": f"F=F2C,F1 S=S L={src} P={P}"}
RESULTS = {}
TAGS = dict(zip(VARIANTS, ("v_base", "v_plus_l2", "v_l2_replaces_l")))


def stack(tag, branches, yn_dk="keep"):
    out = OUT / tag
    sh(["python", "stack_ensemble.py", "--pred-dir", PRED, "--split", "trdev", "--branches", branches,
        "--human-fr-dk", "avoid", "--human-yn-dk", yn_dk, "--c-grid", C["C_GRID"], "--auto-submission", auto_sub,
        "--output-dir", out], capture=True)
    return json.loads((out / "results.json").read_text())


for name, branches in VARIANTS.items():
    RESULTS[name] = stack(TAGS[name], branches)

rows = ["| Biến thể | Cách gộp | YN | FR | FB | CO | **Human cv** | Nhóm A | Nhóm B | Final cv |", "|---|---|---|---|---|---|---|---|---|---|"]
best = None
for name, result in RESULTS.items():
    for method in ("weighted", "hybrid nested"):
        scores, groups = result["results"][method], result["groups"][method]
        human = sum(scores.values()) / 4
        rows.append(f"| {name} | {method} | " + " | ".join(f"{scores[t]:.3f}" for t in ("YN", "FR", "FB", "CO"))
                    + f" | **{human:.4f}** | {groups['A']:.3f} | {groups['B']:.3f} | {(human + auto_cv) / 2:.4f} |")
        if best is None or human > best[2]:
            best = (name, method, human)
text = "\n".join(rows) + f"\n\n**Tốt nhất:** {best[0]} — {best[1]} (Human cv {best[2]:.4f})."
display(Markdown(text))
(OUT / "summary.md").write_text(text + "\n", encoding="utf-8")
to_drive()

$ python crossfit.py combine --pred-dir /content/VLSP-Task5/outputs/predictions_e7 --name trdev
F1: 613 predictions -> human_trdev.jsonl
F2C: 613 predictions -> human_trdev.jsonl
LCOT: 613 predictions -> human_trdev.jsonl
LCOT2: 613 predictions -> human_trdev.jsonl
LP: 597 predictions -> human_trdev.jsonl
LPT: 597 predictions -> human_trdev.jsonl
LPX: 597 predictions -> human_trdev.jsonl
LPY: 597 predictions -> human_trdev.jsonl
S: 613 predictions -> human_trdev.jsonl
Wrote Data/splits/human_trdev.json (30 train + 7 dev stories)
[exit 0, 0.0 phút]
$ python stack_ensemble.py --pred-dir /content/VLSP-Task5/outputs/predictions_e7 --split trdev --branches F=F2C,F1 S=S L=LCOT P=LPY --human-fr-dk avoid --human-yn-dk keep --c-grid 0.1,1,10 --auto-submission /content/drive/MyDrive/VLSP2026/outputs/e4/trdev_LPX/F+S+L+P/auto_submission.json --output-dir /content/VLSP-Task5/outputs/e7/v_base
[exit 0, 1.1 phút]
$ python stack_ensemble.py --pred-dir /content/VLSP-Task5/outputs/predictions_e7 --spli

| Biến thể | Cách gộp | YN | FR | FB | CO | **Human cv** | Nhóm A | Nhóm B | Final cv |
|---|---|---|---|---|---|---|---|---|---|
| base (không L2) | weighted | 0.851 | 0.785 | 0.890 | 0.805 | **0.8328** | 0.910 | 0.688 | 0.9100 |
| base (không L2) | hybrid nested | 0.863 | 0.772 | 0.896 | 0.799 | **0.8325** | 0.908 | 0.690 | 0.9099 |
| +L2 (nhánh M) | weighted | 0.851 | 0.765 | 0.890 | 0.805 | **0.8278** | 0.908 | 0.679 | 0.9075 |
| +L2 (nhánh M) | hybrid nested | 0.876 | 0.752 | 0.903 | 0.805 | **0.8339** | 0.908 | 0.694 | 0.9106 |
| L2 thay L | weighted | 0.820 | 0.725 | 0.883 | 0.785 | **0.8033** | 0.883 | 0.656 | 0.8953 |
| L2 thay L | hybrid nested | 0.820 | 0.725 | 0.890 | 0.805 | **0.8099** | 0.888 | 0.665 | 0.8986 |

**Tốt nhất:** +L2 (nhánh M) — hybrid nested (Human cv 0.8339).

## 6. File nộp của biến thể tốt nhất
Nếu L2 không giúp (biến thể `base` thắng) thì file nộp trùng với E6; khi đó không cần nộp lại.

In [7]:
name, method, human = best
branches = VARIANTS[name]
final_dir = OUT / "submission"
if method == "weighted":
    sh(["python", "compare_branches.py", "--pred-dir", PRED, "--output-dir", OUT / "compare_best", "--branches", branches,
        "--tune-splits", "human=trdev", "--human-fr-dk", "avoid", "--human-yn-dk", C["HUMAN_YN_DK_FINAL"], "--no-use-indifinite"])
    full = "+".join(b.split("=")[0] for b in branches.split())      # luôn dùng tổ hợp đủ các nhánh, như E6
    shutil.copytree(OUT / "compare_best" / full, final_dir, dirs_exist_ok=True)
else:
    stack("final", branches, C["HUMAN_YN_DK_FINAL"])
    shutil.copytree(OUT / "final" / "hybrid", final_dir, dirs_exist_ok=True)
sh(["python", "validate_submission.py", final_dir / "human_submission.json", "--reference", "Data/human_public_test.json"])
sh(["python", "validate_submission.py", final_dir / "auto_submission.json", "--reference", "Data/auto_public_test.json"])
to_drive()
print(f"Bài nộp: {name} — {method} (Human cv {human:.4f}) -> Drive outputs/e7/submission/. Tổng {(time.time() - START) / 60:.1f} phút")

$ python stack_ensemble.py --pred-dir /content/VLSP-Task5/outputs/predictions_e7 --split trdev --branches F=F2C,F1 S=S L=LCOT P=LPY M=LCOT2 --human-fr-dk avoid --human-yn-dk no --c-grid 0.1,1,10 --auto-submission /content/drive/MyDrive/VLSP2026/outputs/e4/trdev_LPX/F+S+L+P/auto_submission.json --output-dir /content/VLSP-Task5/outputs/e7/final
[exit 0, 3.3 phút]
$ python validate_submission.py /content/VLSP-Task5/outputs/e7/submission/human_submission.json --reference Data/human_public_test.json
OK: 132 questions, structure preserved, every answer valid.
[exit 0, 0.0 phút]
$ python validate_submission.py /content/VLSP-Task5/outputs/e7/submission/auto_submission.json --reference Data/auto_public_test.json
OK: 14981 questions, structure preserved, every answer valid.
[exit 0, 0.0 phút]
Bài nộp: +L2 (nhánh M) — hybrid nested (Human cv 0.8339) -> Drive outputs/e7/submission/. Tổng 124.1 phút
